# Step 7 — Station x Month x Time-Period Summary (peak/off-peak/weekend, per-station)

**Input**: `output/processed_hourly_bike_counts_time_categories.csv.gz` (Step 4 output,
per-station per-hour, `time_period` already computed, untouched here)

**Output**:
- `output/station_month_time_period_summary.csv` (328 stations x 14 months x 3 time_periods,
  descriptive statistics)
- `output/station_month_temporal_ratios.csv` (328 stations x 14 months, peak/off-peak/weekend
  ratios)

## Why this step exists

Step 5 (`5_Monthly_time_series_summary.ipynb`) already computed `time_period` statistics, but as
a **network-wide** version with all 328 stations combined -- it can only answer "what does the
whole Hamburg cycling network's commute peak/off-peak/weekend pattern look like." This step
answers "which station, which month" -- the same logic already applied to the day/night side
(Step 5 -> Step 6's `station_month_day_night.csv`), just for a different dimension (time_period
instead of day/night).

## Decisions on record (consistent with the project discussion)

1. **`time_period` stays as three categories** (`weekday_peak` / `weekday_offpeak` /
   `weekend_public_holiday`), not split into four. The earlier `4_timeseries_check.ipynb`'s
   `categorize_time()` was checked and does have four categories (Public Holiday kept separate),
   but that code determines peak hours directly from `hour_utc.hour` without a timezone
   conversion -- effectively judging local peak hours using UTC, an unfixed bug -- so it isn't a
   good template to copy. Three categories are confirmed sufficient for the current analysis goal;
   four categories can be revisited later if needed.
2. **Timezone unchanged**: `month`/`time_period` carry over from Step 4's columns, used as Hamburg
   local time with no conversion (consistent with every earlier step).
3. **Covariance is not computed here**: the definition found in the earlier code was "the lag-1
   autocovariance between adjacent rows within the same time_category, after filtering"
   (`group['bike_count_hourly'].cov(...shift(1))`) -- not covariance with an external variable,
   and rows adjacent after filtering aren't necessarily adjacent in real time, so the metric's
   meaning is fairly unclear. Left out of this step.
4. **Low-sample handling**: computing `peak_offpeak_ratio` / `weekday_weekend_ratio` runs into the
   same class of problem as the day/night side -- some stations have too few (sometimes zero)
   total rows in `weekday_peak`, `weekday_offpeak`, or `weekend_public_holiday` in a given month,
   and dividing directly produces unreasonable extreme values or `inf`. The same approach already
   validated in Steps 5/6 is reused: if either side has `total_count < 10`, that ratio is flagged
   unreliable (`ratio_reliable = False`); the value is kept but should be excluded or checked
   individually downstream.


In [1]:
import pandas as pd
import numpy as np

PROCESSED_PATH = "output/processed_hourly_bike_counts_time_categories.csv.gz"
STATION_META_PATH = "output/station_metadata.csv"
OUT_SUMMARY = "output/station_month_time_period_summary.csv"
OUT_RATIOS = "output/station_month_temporal_ratios.csv"

RELIABILITY_MIN_SAMPLE = 10  # same threshold used on the day/night side

In [2]:
df = pd.read_csv(
    PROCESSED_PATH,
    dtype={'station_id': 'int64', 'month': 'str', 'time_period': 'str', 'bike_count_hourly': 'float64'},
    usecols=['station_id', 'month', 'time_period', 'bike_count_hourly'],
)
station_meta = pd.read_csv(STATION_META_PATH)
print(f"Loaded: {len(df):,} rows, {df['station_id'].nunique()} stations")

Loaded: 3,227,633 rows, 328 stations


## station x month x time_period statistics

In [3]:
s = df.groupby(['station_id', 'month', 'time_period']).agg(
    total_count=('bike_count_hourly', 'sum'),
    number_of_records=('bike_count_hourly', 'size'),
    average_hourly_count=('bike_count_hourly', 'mean'),
    median_hourly_count=('bike_count_hourly', 'median'),
    standard_deviation=('bike_count_hourly', 'std'),
).reset_index()
s['coefficient_of_variation'] = s['standard_deviation'] / s['average_hourly_count']

s = s.merge(station_meta[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84']],
            on='station_id', how='left')
s = s[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84', 'month', 'time_period',
       'total_count', 'number_of_records', 'average_hourly_count', 'median_hourly_count',
       'standard_deviation', 'coefficient_of_variation']]
s = s.sort_values(['station_id', 'month', 'time_period']).reset_index(drop=True)

s.to_csv(OUT_SUMMARY, index=False)
theoretical_max = df['station_id'].nunique() * df['month'].nunique() * df['time_period'].nunique()
print(f"Written: {OUT_SUMMARY}")
print(f"Rows: {len(s):,} (theoretical max {theoretical_max:,})")
s.head()

Written: output/station_month_time_period_summary.csv
Rows: 13,383 (theoretical max 13,776)


,station_id,station_name,longitude_wgs84,latitude_wgs84,month,time_period,total_count,number_of_records,average_hourly_count,median_hourly_count,standard_deviation,coefficient_of_variation
0,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-01,weekday_offpeak,8216.0,396,20.747475,12.0,20.882356,1.006501
1,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-01,weekday_peak,13394.0,132,101.469697,68.5,82.416966,0.812232
2,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-01,weekend_public_holiday,3858.0,216,17.861111,6.0,25.077168,1.404009
3,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-02,weekday_offpeak,8282.0,360,23.005556,13.0,22.240393,0.966740
4,5677,Zählfeld J_36.1_1_I (veraltet),9.999514,53.580505,2025-02,weekday_peak,13629.0,120,113.575000,79.5,89.065519,0.784200


## station x month ratios (peak/off-peak/weekend)

In [4]:
pivot_avg = s.pivot(index=['station_id', 'month'], columns='time_period', values='average_hourly_count')
pivot_total = s.pivot(index=['station_id', 'month'], columns='time_period', values='total_count')
pivot_n = s.pivot(index=['station_id', 'month'], columns='time_period', values='number_of_records')

ratios = pd.DataFrame(index=pivot_avg.index)

# peak_offpeak_ratio
with np.errstate(divide='ignore', invalid='ignore'):
    ratios['peak_offpeak_ratio'] = pivot_avg['weekday_peak'] / pivot_avg['weekday_offpeak']
ratios['peak_offpeak_ratio'] = ratios['peak_offpeak_ratio'].replace([np.inf, -np.inf], np.nan)

# weekday_weekend_ratio: weekday = weekday_peak + weekday_offpeak, weighted combination
weekday_total = pivot_total['weekday_peak'].fillna(0) + pivot_total['weekday_offpeak'].fillna(0)
weekday_n = pivot_n['weekday_peak'].fillna(0) + pivot_n['weekday_offpeak'].fillna(0)
weekday_avg = weekday_total / weekday_n
with np.errstate(divide='ignore', invalid='ignore'):
    ratios['weekday_weekend_ratio'] = weekday_avg / pivot_avg['weekend_public_holiday']
ratios['weekday_weekend_ratio'] = ratios['weekday_weekend_ratio'].replace([np.inf, -np.inf], np.nan)

# peak_share / weekend_share (denominator = that station-month's total across all 3 categories)
monthly_total = pivot_total.sum(axis=1, skipna=True)
ratios['peak_share'] = pivot_total['weekday_peak'] / monthly_total
ratios['weekend_share'] = pivot_total['weekend_public_holiday'] / monthly_total

# Reliability: unreliable if any of peak/off-peak/weekend has total_count < 10 (same logic as day/night)
low_sample = (
    (pivot_total['weekday_peak'].fillna(0) < RELIABILITY_MIN_SAMPLE) |
    (pivot_total['weekday_offpeak'].fillna(0) < RELIABILITY_MIN_SAMPLE) |
    (pivot_total['weekend_public_holiday'].fillna(0) < RELIABILITY_MIN_SAMPLE)
)
ratios['ratio_reliable'] = ~low_sample
ratios.loc[low_sample, ['peak_offpeak_ratio', 'weekday_weekend_ratio']] = np.nan

ratios = ratios.reset_index()
ratios = ratios.merge(station_meta[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84']],
                       on='station_id', how='left')
ratios = ratios[['station_id', 'station_name', 'longitude_wgs84', 'latitude_wgs84', 'month',
                  'peak_offpeak_ratio', 'weekday_weekend_ratio', 'peak_share', 'weekend_share',
                  'ratio_reliable']]
ratios = ratios.sort_values(['station_id', 'month']).reset_index(drop=True)

ratios.to_csv(OUT_RATIOS, index=False)
print(f"Written: {OUT_RATIOS}")
print(f"Rows: {len(ratios):,}, flagged unreliable: {(~ratios['ratio_reliable']).sum():,}")
ratios.describe()

Written: output/station_month_temporal_ratios.csv
Rows: 4,461, flagged unreliable: 152


,station_id,longitude_wgs84,latitude_wgs84,peak_offpeak_ratio,weekday_weekend_ratio,peak_share,weekend_share
count,4461.000000,4461.000000,4461.000000,4309.000000,4309.000000,4460.000000,4460.000000
mean,7422.578570,10.005901,53.564909,1.967309,1.803810,0.303053,0.214002
std,1598.519293,0.070036,0.048350,0.702360,0.530515,0.074661,0.063704
min,5677.000000,9.811806,53.456430,0.465209,0.125884,0.000000,0.000000
25%,5775.000000,9.970350,53.548740,1.492934,1.453707,0.252912,0.174972
50%,7771.000000,9.998648,53.564580,1.783784,1.737324,0.291159,0.205261
75%,8486.000000,10.036357,53.595016,2.282178,2.100565,0.347450,0.241247
max,11965.000000,10.207702,53.656773,6.295178,4.968603,0.857143,0.876106


## Sanity check: are inf / extreme values cleaned up?

In [5]:
for col in ['peak_offpeak_ratio', 'weekday_weekend_ratio']:
    n_inf = np.isinf(ratios[col]).sum()
    print(f"{col}: inf={n_inf}, max(reliable only)={ratios.loc[ratios['ratio_reliable'], col].max():.2f}")

peak_offpeak_ratio: inf=0, max(reliable only)=6.30
weekday_weekend_ratio: inf=0, max(reliable only)=4.97
